# 🧬 F1 (R twin) — Sequence Analysis Toolkit · *build notebook*

The **R version** of the same toolkit you built in Python. You already know the logic, so this is about learning **R idioms** — where R does things differently from Python. That contrast is exactly what the dual-language rule is for.

**This is YOUR build notebook** — the `# TODO`s are yours to fill in.

**Platform:** **Kaggle** (New Notebook → set language to **R**), or any R + Jupyter setup. Not Colab (Colab is Python-first).

**Watch for these R-vs-Python differences as you go:**
- R is **1-indexed**, and ranges are **inclusive**: `substr(x, i, i+2)` takes 3 characters.
- A string is **not** iterable character-by-character — split it first: `strsplit(x, '')[[1]]` gives a character vector.
- R uses **named vectors / lists** instead of Python dicts: `comp['A']`.
- Assignment is `<-`; join strings with `paste(..., collapse='')`; count with `table()`.

## 1–2. Research question & background

Same as the Python project: for the human insulin (**INS**) mRNA (`NM_000207`), compute GC content, GC skew, k-mer frequencies, the reverse complement, ORFs, and the protein translation — from scratch, cross-checked against a library (here **seqinr**).

(The biology is unchanged from your Python notebook, so we won't repeat it — focus on the R.)

## 3. Documentation drill (R packages)

We use two **CRAN** packages (easy to install on Kaggle):
- **rentrez** — R client for the NCBI Entrez API. Docs: https://docs.ropensci.org/rentrez/
- **seqinr** — classic R sequence-analysis toolkit (our cross-check). Docs: https://seqinr.r-forge.r-project.org/

Answer in the next cell as comments (read, don't code yet):
1. In `rentrez`, which function fetches a record? What do its `db`, `id`, `rettype` arguments do?
2. In `seqinr`: which function computes GC content? which counts k-mer words? which translates DNA?
3. How do you get the complement of a sequence in seqinr, and how would you make it a *reverse* complement?

In [ ]:
# Documentation-drill answers (comments only):
# 1. rentrez fetch function: ____ ; db: ____ , id: ____ , rettype: ____
# 2. seqinr -> GC content: ____ ; k-mer counts: ____ ; translate: ____
# 3. seqinr complement: ____ ; reverse complement = ____


In [ ]:
# --- Setup: install the two CRAN packages (first run takes a minute) ---
# WHY: rentrez = Entrez API client; seqinr = sequence utilities for cross-checking.
if (!requireNamespace("rentrez", quietly = TRUE)) install.packages("rentrez")
if (!requireNamespace("seqinr",  quietly = TRUE)) install.packages("seqinr")
library(rentrez)
library(seqinr)
dir.create("data",    showWarnings = FALSE)
dir.create("results", showWarnings = FALSE)
cat("packages loaded\n")

## 3b. Fetch the data from the NCBI Entrez API

`entrez_fetch` returns the record as one big FASTA **string**. We save it, then parse it.

In [ ]:
accession <- "NM_000207"   # human insulin (INS) mRNA
fasta_txt <- entrez_fetch(db = "nuccore", id = accession, rettype = "fasta", retmode = "text")
writeLines(fasta_txt, "data/sequence.fasta")
cat(substr(fasta_txt, 1, 120), "...\n")

## 5. Analysis pipeline

Work top to bottom. Predict each output before running.

### TODO 1 — Parse the FASTA from scratch
R note: `readLines()` gives a character vector (one element per line). Header lines start with `>`.

In [ ]:
parse_fasta <- function(path) {
  lines <- readLines(path)
  seqs <- list()            # R's version of a dict
  current_id <- NULL
  for (line in lines) {
    line <- trimws(line)
    if (nchar(line) == 0) next
    if (startsWith(line, ">")) {
      # TODO: current_id <- the header without '>', first word only
      #       Hint: strsplit(substring(line, 2), " ")[[1]][1]
      #       then seqs[[current_id]] <- ""
      # YOUR CODE HERE
    } else {
      # TODO: append this line to seqs[[current_id]]
      #       Hint: paste0(seqs[[current_id]], line)
      # YOUR CODE HERE
    }
  }
  seqs
}

records <- parse_fasta("data/sequence.fasta")
seq <- records[[1]]
cat("records:", length(records), "| length:", nchar(seq), "\n")

### TODO 2 — GC content
R note: split the string to a character vector, then compare. `sum(v == 'G')` counts G's.

In [ ]:
gc_content <- function(s) {
  s <- toupper(s)
  chars <- strsplit(s, "")[[1]]
  if (length(chars) == 0) return(0)
  # TODO: (number of G + number of C) / length
  # YOUR CODE HERE
}

cat("GC =", round(gc_content(seq), 3), "\n")
# cross-check with seqinr::GC (expects a lowercase character vector):
cat("seqinr GC =", round(GC(strsplit(tolower(seq), "")[[1]]), 3), "\n")

### TODO 3 — GC skew (non-overlapping windows)
R note: `seq(1, n - window + 1, by = window)` gives 1-based window starts; `substr(s, i, i+window-1)` takes the window.

In [ ]:
gc_skew <- function(s, window = 50) {
  s <- toupper(s); n <- nchar(s); skews <- c()
  for (i in seq(1, n - window + 1, by = window)) {
    w <- substr(s, i, i + window - 1)
    v <- strsplit(w, "")[[1]]
    g <- sum(v == "G"); c <- sum(v == "C")
    # TODO: append (g - c)/(g + c) to skews, or 0 if g + c == 0
    # Hint: skews <- c(skews, value)
    # YOUR CODE HERE
  }
  skews
}
skews <- gc_skew(seq, 50)
cat("windows:", length(skews), "\n"); print(round(skews, 3))

### TODO 4 — k-mer counts
R note: build all length-k substrings, then `table()` counts them.

In [ ]:
kmer_counts <- function(s, k = 3) {
  s <- toupper(s); n <- nchar(s)
  # TODO: kmers <- sapply(1:(n-k+1), function(i) substr(s, i, i+k-1)); then table(kmers)
  # YOUR CODE HERE
}
counts <- kmer_counts(seq, 3)
cat("distinct 3-mers:", length(counts), "\n")
print(head(sort(counts, decreasing = TRUE), 5))

### TODO 5 — reverse complement
R note: map bases with a **named vector**, then `rev()` and `paste(collapse='')`. Self-check: revcomp twice == original.

In [ ]:
reverse_complement <- function(s) {
  s <- toupper(s)
  comp <- c(A = "T", T = "A", G = "C", C = "G", N = "N")   # named vector = lookup table
  chars <- strsplit(s, "")[[1]]
  # TODO: complemented <- comp[chars]; then reverse and paste into one string
  # Hint: paste(rev(complemented), collapse = "")
  # YOUR CODE HERE
}
cat("ATGC ->", reverse_complement("ATGC"), "(expect GCAT)\n")
cat("involution:", reverse_complement(reverse_complement(seq)) == seq, "\n")

### TODO 6 — find ORFs (3 forward frames)  *(the hard one)*
R note: **1-based** indices and inclusive `substr`. A codon at start `i` is `substr(s, i, i+2)`.

In [ ]:
stops <- c("TAA", "TAG", "TGA")
find_orfs <- function(s, min_len = 60) {
  s <- toupper(s); n <- nchar(s); orfs <- c()
  for (frame in 0:2) {
    i <- 1 + frame
    while (i <= n - 2) {
      if (substr(s, i, i + 2) == "ATG") {
        j <- i
        while (j <= n - 2) {
          if (substr(s, j, j + 2) %in% stops) {
            orf <- substr(s, i, j + 2)
            # TODO: if nchar(orf) >= min_len, append orf to orfs (orfs <- c(orfs, orf))
            # YOUR CODE HERE
            i <- j
            break
          }
          j <- j + 3
        }
      }
      i <- i + 3
    }
  }
  orfs
}
orfs <- find_orfs(seq, 60)
cat("ORFs found:", length(orfs), "\n")
for (o in orfs) cat(nchar(o), "bp\n")

### TODO 7 — translate DNA -> protein
The codon table is a **named vector** (given). Loop over codons, look each up.

In [ ]:
CODON <- c(
 TTT="F",TTC="F",TTA="L",TTG="L",CTT="L",CTC="L",CTA="L",CTG="L",
 ATT="I",ATC="I",ATA="I",ATG="M",GTT="V",GTC="V",GTA="V",GTG="V",
 TCT="S",TCC="S",TCA="S",TCG="S",CCT="P",CCC="P",CCA="P",CCG="P",
 ACT="T",ACC="T",ACA="T",ACG="T",GCT="A",GCC="A",GCA="A",GCG="A",
 TAT="Y",TAC="Y",TAA="*",TAG="*",CAT="H",CAC="H",CAA="Q",CAG="Q",
 AAT="N",AAC="N",AAA="K",AAG="K",GAT="D",GAC="D",GAA="E",GAG="E",
 TGT="C",TGC="C",TGA="*",TGG="W",CGT="R",CGC="R",CGA="R",CGG="R",
 AGT="S",AGC="S",AGA="R",AGG="R",GGT="G",GGC="G",GGA="G",GGG="G")

translate_dna <- function(dna) {
  dna <- toupper(dna); protein <- ""
  for (i in seq(1, nchar(dna) - 2, by = 3)) {
    codon <- substr(dna, i, i + 2)
    # TODO: aa <- CODON[[codon]] (use 'X' if not found); protein <- paste0(protein, aa)
    # YOUR CODE HERE
  }
  protein
}
longest <- orfs[[which.max(nchar(orfs))]]
cat("mine    :", translate_dna(longest), "\n")
# cross-check with seqinr::translate (needs a lowercase character vector):
theirs <- paste(translate(strsplit(tolower(longest), "")[[1]]), collapse = "")
cat("seqinr  :", theirs, "\n")

## 6–7. Results (CSV) + figure

In [ ]:
top3 <- names(sort(kmer_counts(seq, 3), decreasing = TRUE))[1]
summary_df <- data.frame(
  seq_id = names(records)[1],
  length = nchar(seq),
  gc_content = round(gc_content(seq), 4),
  n_orfs = length(orfs),
  longest_orf_bp = max(nchar(orfs)),
  top_3mer = top3
)
write.csv(summary_df, "results/summary_R.csv", row.names = FALSE)
print(summary_df)

png("results/gc_skew_R.png", width = 1400, height = 800, res = 200)
plot(skews, type = "o", xlab = "window index (50 bp)",
     ylab = "GC skew (G-C)/(G+C)", main = paste("GC skew across", accession))
abline(h = 0, col = "grey")
dev.off()
cat("saved results/summary_R.csv and results/gc_skew_R.png\n")

## 8–9. Limitations & interpretation *(write in your own words)*

- Compare your R results to your Python results — do GC content, the ORF length, and the translation **match**? They should. If they differ, that's a bug worth finding (this is the point of doing both).
- **TODO:** 2–3 sentences interpreting the result, and 1–2 limitations (same as Python: forward-strand ORFs only, single sequence, skew not meaningful at mRNA scale).

## 11. Abstract + commit
Once it runs clean, this R notebook goes in the **same repo** as the Python one, and the README gets a short "Python vs R" comparison. Download it from Kaggle and drop it into `D:\GIT HUB PROJECTS\sequence-analysis-toolkit\`.